# Transcribe and evaluate Dutch audio with Gemma 4 using MLX-VLM

First run `uv sync --group mlx-vlm` to install the MLX-VLM dependencies in the project's shared `.venv`.
Select `.venv/bin/python` as this notebook's kernel, then run all cells from the repository root. `ffmpeg` must be on PATH.

The notebook loads `google/gemma-4-E4B-it` once through MLX-VLM on Apple Silicon. It decodes the UMC Utrecht MP3 to mono 16 kHz WAV and splits it into chunks of at most 28 seconds, preferring pauses, to stay under Gemma's 30-second audio limit.
Chunks do not overlap; a word may be split when no suitable pause is found. Each chunk is transcribed locally with thinking disabled and temperature zero, then the text is combined in order.

The final cell prints **word error rate (WER)** against both curated references, using exactly the Whisper notebook's normalization and scoring. Reference text is used only for scoring, never in the transcription prompt.

Reference: [MLX-VLM Gemma audio documentation](https://github.com/Blaizzy/mlx-vlm/blob/main/mlx_vlm/models/gemma4/README.md).


In [1]:
import io
import re
import shutil
import subprocess
import tempfile
import time
import wave
from pathlib import Path

import mlx.core as mx
from mlx_vlm import generate, load
from mlx_vlm.prompt_utils import apply_chat_template

MODEL_NAME = "google/gemma-4-E4B-it"
DATA_DIR = Path("data/UMC Utrecht - Jeroen Hendrikse - radiologieverslag")
AUDIO_PATH = DATA_DIR / f"{DATA_DIR.name}.mp3"
CURATED_TXT_PATH = DATA_DIR / f"{DATA_DIR.name} - curated.txt"
CURATED_SRT_PATH = DATA_DIR / f"{DATA_DIR.name} - curated.srt"
CHUNK_SECONDS = 28
MAX_TOKENS = 1024  # Output budget per chunk; truncated responses raise an error.
TRANSCRIPTION_PROMPT = (
    "Transcribe the following Dutch speech segment verbatim into Dutch text. "
    "Return only the transcription, without explanations, headings, or timestamps. "
    "Preserve repetitions, hesitations, and filler words. Do not summarize or translate."
)

for path in (AUDIO_PATH, CURATED_TXT_PATH, CURATED_SRT_PATH):
    if not path.is_file():
        raise FileNotFoundError(
            f"File not found: {path.resolve()}. Run from the repository root."
        )
if shutil.which("ffmpeg") is None:
    raise RuntimeError(
        "Install ffmpeg and make it available on PATH before running this notebook."
    )
if not 0 < CHUNK_SECONDS <= 28:
    raise ValueError("CHUNK_SECONDS must be between 0 and 28 seconds.")

In [2]:
# Avoid download progress widgets that require external JavaScript in the notebook UI.
from huggingface_hub.utils import disable_progress_bars

disable_progress_bars()

started = time.perf_counter()
model, processor = load(MODEL_NAME)
prompt = apply_chat_template(
    processor,
    model.config,
    TRANSCRIPTION_PROMPT,
    num_audios=1,
    chat_template_kwargs={"enable_thinking": False},
)
print(f"Loaded {MODEL_NAME} in {time.perf_counter() - started:.1f}s")

/Users/coert/workspace/workspace-python/workshop-local-llm/.venv/lib/python3.14/site-packages/transformers/audio_utils.py:829: UserWarning: At least one mel filter has all zero values. The value for `num_mel_filters` (128) may be set too high. Or, the value for `num_frequency_bins` (257) may be set too low.
  warnings.warn(


Loaded google/gemma-4-E4B-it in 10.9s


In [3]:
# Decode and detect pauses locally; inference runs directly through MLX-VLM.
decoded = subprocess.run(
    [
        "ffmpeg",
        "-hide_banner",
        "-nostdin",
        "-i",
        str(AUDIO_PATH),
        "-vn",
        "-ac",
        "1",
        "-ar",
        "16000",
        "-af",
        "silencedetect=noise=-35dB:d=0.2",
        "-f",
        "wav",
        "-acodec",
        "pcm_s16le",
        "pipe:1",
    ],
    check=False,
    capture_output=True,
)
if decoded.returncode:
    raise RuntimeError(decoded.stderr.decode(errors="replace"))

# ffmpeg logs each silence's end and duration; split at its midpoint.
pauses = [
    float(end) - float(duration) / 2
    for end, duration in re.findall(
        r"silence_end: ([0-9.]+) \| silence_duration: ([0-9.]+)",
        decoded.stderr.decode(errors="replace"),
    )
]
with wave.open(io.BytesIO(decoded.stdout), "rb") as audio:
    sample_rate = audio.getframerate()
    sample_width = audio.getsampwidth()
    pcm = audio.readframes(audio.getnframes())
frame_count = len(pcm) // sample_width
max_frames = int(CHUNK_SECONDS * sample_rate)
chunks = []
start = 0
while start < frame_count:
    end = min(start + max_frames, frame_count)
    if end < frame_count:
        candidates = [
            int(p * sample_rate)
            for p in pauses
            if start + max_frames * 0.7 < p * sample_rate <= end
        ]
        if candidates:
            end = max(candidates)
    buffer = io.BytesIO()
    with wave.open(buffer, "wb") as chunk:
        chunk.setnchannels(1)
        chunk.setsampwidth(sample_width)
        chunk.setframerate(sample_rate)
        chunk.writeframes(pcm[start * sample_width : end * sample_width])
    chunks.append(
        {
            "start": start / sample_rate,
            "end": end / sample_rate,
            "audio": buffer.getvalue(),
        }
    )
    start = end
print(f"Audio: {frame_count / sample_rate:.1f}s; {len(chunks)} chunks", flush=True)

chunk_results = []
transcription_started = time.perf_counter()
with tempfile.TemporaryDirectory(prefix="gemma-audio-") as temporary_dir:
    for index, chunk in enumerate(chunks, start=1):
        chunk_path = Path(temporary_dir) / f"chunk-{index:03d}.wav"
        chunk_path.write_bytes(chunk["audio"])
        started = time.perf_counter()
        result = generate(
            model=model,
            processor=processor,
            prompt=prompt,
            audio=[str(chunk_path)],
            max_tokens=MAX_TOKENS,
            temperature=0.0,
            verbose=False,
        )
        text = result.text.strip()
        if result.generation_tokens >= MAX_TOKENS:
            raise RuntimeError(
                f"Chunk {index} reached MAX_TOKENS; increase the budget before scoring."
            )
        if not text:
            raise RuntimeError(f"Chunk {index} returned no transcription.")
        elapsed = time.perf_counter() - started
        chunk_results.append(
            {
                "start": chunk["start"],
                "end": chunk["end"],
                "text": text,
                "elapsed_seconds": elapsed,
            }
        )
        print(
            f"Chunk {index}/{len(chunks)} ({chunk['start']:.1f}–{chunk['end']:.1f}s): {elapsed:.1f}s",
            flush=True,
        )
        mx.clear_cache()

transcription = " ".join(chunk["text"] for chunk in chunk_results)
print(f"Transcription completed in {time.perf_counter() - transcription_started:.1f}s")

Audio: 473.6s; 18 chunks
Chunk 1/18 (0.0–26.3s): 13.2s
Chunk 2/18 (26.3–48.4s): 7.3s
Chunk 3/18 (48.4–75.4s): 7.2s
Chunk 4/18 (75.4–102.1s): 8.9s
Chunk 5/18 (102.1–129.2s): 8.3s
Chunk 6/18 (129.2–155.9s): 7.2s
Chunk 7/18 (155.9–183.3s): 6.9s
Chunk 8/18 (183.3–209.2s): 7.6s
Chunk 9/18 (209.2–235.3s): 7.7s
Chunk 10/18 (235.3–257.7s): 6.9s
Chunk 11/18 (257.7–285.1s): 8.8s
Chunk 12/18 (285.1–312.6s): 9.5s
Chunk 13/18 (312.6–338.9s): 8.6s
Chunk 14/18 (338.9–365.3s): 8.7s
Chunk 15/18 (365.3–392.6s): 8.4s
Chunk 16/18 (392.6–419.4s): 8.2s
Chunk 17/18 (419.4–446.4s): 8.7s
Chunk 18/18 (446.4–473.6s): 6.7s
Transcription completed in 148.9s


In [4]:
print(transcription)

Ja, dat zou moeten werken. Ja. Oké, dus wat er eigenlijk kijkt in de verslaglijst en ik zie een Emery hersen eh staan van een bepaalde aanvraag. Ik weet dat het een hersen tumor follow-up waarschijnlijk zal zijn. Dus ik klik die patiënt aan en zie dan inderdaad een follow-up staan van een patiënt met drie maanden geleden ook een scan, dus ik heb even Heel snel zie ik het scanlijstje met de verschillende data van van eerdere onderzoeken. Daarin zie je bijvoorbeeld dat er een jaar lang al follow-up wordt gemaakt. Dus een jaar geleden was blijkbaar de ontdekking van de tumor met een aantal onderzoeken extern. Ik ik kijk daar later naar. Ik heb het in mijn geheugen opgeslagen van oké, als ik de uitgangssituatie wil weten, moet ik de scan van een jaar terug. De nastheringen nu is eigenlijk alleen relevant follow-up. Oké, verandert er iets ten opzichte van drie maanden terug? Dat is eigenlijk de uitgangsdata, gewoon de grove data van momenten dat ik kan gaan kijken, dan hang ik ze voor ga ik

## Compare with curated transcriptions

WER = (substitutions + deletions + insertions) / reference word count.
We use minimum word-level edit distance after Unicode normalization, case folding, and removal of punctuation; hyphens split words.
Accents, numbers, repeated words, and fillers are retained. SRT cue numbers and timestamps are excluded.
This measures transcription differences, including any editorial changes in the curated references.


In [5]:
import re
import unicodedata


def srt_text(contents):
    """Extract subtitle text, excluding each cue's index and timing line."""
    cues = re.split(r"\n\s*\n", contents.strip())
    text = []
    for cue in cues:
        lines = cue.splitlines()
        timing = next((i for i, line in enumerate(lines) if "-->" in line), None)
        if timing is None:
            raise ValueError(f"Subtitle cue has no timestamp: {cue!r}")
        text.extend(lines[timing + 1 :])
    return " ".join(text)


def normalize_words(text):
    return re.findall(r"[^\W_]+", unicodedata.normalize("NFKC", text).casefold())


def word_errors(reference, hypothesis):
    """Levenshtein alignment; ties prefer fewer substitutions, then deletions."""
    # Each tuple stores (total errors, substitutions, deletions, insertions).
    previous = [(i, 0, 0, i) for i in range(len(hypothesis) + 1)]
    for i, reference_word in enumerate(reference, start=1):
        current = [(i, 0, i, 0)]
        for j, hypothesis_word in enumerate(hypothesis, start=1):
            if reference_word == hypothesis_word:
                current.append(previous[j - 1])
                continue
            cost, substitutions, deletions, insertions = previous[j - 1]
            substitute = (cost + 1, substitutions + 1, deletions, insertions)
            cost, substitutions, deletions, insertions = previous[j]
            delete = (cost + 1, substitutions, deletions + 1, insertions)
            cost, substitutions, deletions, insertions = current[j - 1]
            insert = (cost + 1, substitutions, deletions, insertions + 1)
            current.append(min(substitute, delete, insert))
        previous = current
    errors, substitutions, deletions, insertions = previous[-1]
    if not reference:
        raise ValueError("The reference contains no words; WER is undefined.")
    return {
        "reference_words": len(reference),
        "substitutions": substitutions,
        "deletions": deletions,
        "insertions": insertions,
        "errors": errors,
        "wer": errors / len(reference),
    }


references = {
    "Curated TXT (primary)": CURATED_TXT_PATH.read_text(encoding="utf-8-sig"),
    "Curated SRT": srt_text(CURATED_SRT_PATH.read_text(encoding="utf-8-sig")),
}

In [6]:
hypothesis_words = normalize_words(transcription)
reference_words = {name: normalize_words(text) for name, text in references.items()}
comparison_results = {
    name: word_errors(words, hypothesis_words)
    for name, words in reference_words.items()
}

print(f"Gemma 4 transcription: {len(hypothesis_words)} normalized words")
print(
    f"Curated TXT and SRT match after normalization: {len({tuple(words) for words in reference_words.values()}) == 1}"
)
for name, result in comparison_results.items():
    print(f"\n{name} — Word error rate (WER): {result['wer']:.2%}")
    print(
        f"  ({result['substitutions']} substitutions + "
        f"{result['deletions']} deletions + {result['insertions']} insertions)"
        f" / {result['reference_words']} reference words"
        f" = {result['errors']} / {result['reference_words']}"
    )

Gemma 4 transcription: 1344 normalized words
Curated TXT and SRT match after normalization: True

Curated TXT (primary) — Word error rate (WER): 7.78%
  (69 substitutions + 21 deletions + 15 insertions) / 1350 reference words = 105 / 1350

Curated SRT — Word error rate (WER): 7.78%
  (69 substitutions + 21 deletions + 15 insertions) / 1350 reference words = 105 / 1350
